# 08 · Grad-CAM

**Reads `folds_infection.csv` from notebook 04. Trains its own model.**

Produces Grad-CAM heatmaps for the infection task and measures how activation
distributes across the colour clusters.

## What this can and cannot claim
The tissue-zone alignment score compares Grad-CAM activation against masks from
the **same LAB k-means that notebook 03 showed does not identify tissue** — it
graded 86.3% of healthy skin as severe, and its luminance ordering is inverted
(granulation L\* 118 sits below periwound callus L\* 168).

So a high alignment score says the model attends to the same dark regions the
clustering labels necrotic, which include shadow, wound depth, hair and darker
skin. It does **not** say the model attends to necrotic tissue. This measures
agreement between two procedures that share a failure mode.

Reported as an internal-consistency diagnostic only. Establishing clinical
correctness needs masks from a segmenter trained on expert tissue annotations,
which is future work.

## A separate model, deliberately
Grad-CAM needs gradients reaching convolutional feature maps. The frozen linear
probe from notebook 07 has no gradient path into the backbone, so this notebook
fine-tunes its own EfficientNet-B0 end-to-end on folds 1–4 and holds out fold 0.
Its numbers are reported separately and never substituted for the AUROC 0.816
control result.

## Area normalisation
Raw activation fraction is confounded by cluster size: a cluster covering half the
image absorbs half the activation by area alone. Cell 7 divides by area to give a
lift ratio, where 1.0 means attention proportional to size and no preferential
targeting.

## Outputs
`gradcam_scores.json`, `gradcam_alignment.csv`, `figures/gradcam/gradcam_panel.{png,pdf}`


In [ ]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')

INTERIM  = Path('data/interim')
FEATURES = Path('data/features')
OUT      = Path('outputs'); OUT.mkdir(exist_ok=True)
FIGDIR   = OUT / 'figures' / 'gradcam'; FIGDIR.mkdir(parents=True, exist_ok=True)

INF_FOLDS = INTERIM / 'folds_infection.csv'
SEED, INPUT_SIZE, N_FOLDS = 42, 224, 5
N_EXAMPLES = 8      # per class, for the figure panel

if not INF_FOLDS.exists():
    print(f'STOPPING. {INF_FOLDS} not found. Run notebook 04 first.')
    raise SystemExit(1)

inf = pd.read_csv(INF_FOLDS)
print(f'{len(inf):,} images, {inf.photo_unit.nunique():,} units, '
      f'{inf.label.sum():,} infected')

In [ ]:
# Cell 2 · fine-tune a small CNN end-to-end for Grad-CAM
# Grad-CAM needs gradients flowing back to convolutional feature maps.
# The frozen linear probe used in notebook 07 has no gradient path into
# the backbone, so a separate end-to-end model is trained here. This is
# a DIFFERENT model from the one that produced AUROC 0.816; its numbers
# are reported separately and never substituted for the control result.
import torch, torch.nn as nn, torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEV = ('cuda' if torch.cuda.is_available()
       else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEV)

_MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
_STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

class ImgDS(Dataset):
    def __init__(s, paths, labels):
        s.p, s.y = list(paths), np.asarray(labels, dtype=np.float32)
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        try:
            with Image.open(s.p[i]) as im:
                im = im.convert('RGB').resize((INPUT_SIZE, INPUT_SIZE))
            a = torch.from_numpy(np.asarray(im, dtype=np.float32)/255.0)
            x = (a.permute(2,0,1) - _MEAN) / _STD
        except Exception:
            x = torch.zeros(3, INPUT_SIZE, INPUT_SIZE)
        return x, s.y[i]

def build_model():
    m = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 1))
    return m.to(DEV)

print('model and dataset defined')

In [ ]:
# Cell 3 · train on folds 1-4, hold out fold 0 for the Grad-CAM examples
# One fold only. Grad-CAM is a qualitative diagnostic, not a performance
# claim, so a full 5-fold run would spend compute without changing what
# the heatmaps show.
import torch.nn.functional as F, time

HOLDOUT = 0
tr = inf[inf.fold != HOLDOUT]
te = inf[inf.fold == HOLDOUT]
print(f'train {len(tr):,} images, holdout {len(te):,} images')

CKPT = OUT / 'gradcam_model.pt'
if CKPT.exists():
    net = build_model()
    net.load_state_dict(torch.load(CKPT, map_location=DEV))
    print(f'loaded cached model from {CKPT.name}')
else:
    net = build_model()
    dl = DataLoader(ImgDS(tr.path, tr.label), batch_size=32, shuffle=True,
                    num_workers=0)   # 0 workers: notebook-defined class
    pos_w = torch.tensor([(tr.label == 0).sum() / max((tr.label == 1).sum(), 1)],
                         dtype=torch.float32).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    use_amp = (DEV == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    EPOCHS = 4
    t0 = time.time()
    for ep in range(EPOCHS):
        net.train(); tot = 0.0
        for x, y in dl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(
                    net(x).squeeze(1), y, pos_weight=pos_w)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += float(loss) * len(x)
        print(f'  epoch {ep}  loss {tot/len(tr):.4f}  ({time.time()-t0:.0f}s)')
    torch.save(net.state_dict(), CKPT)
    print(f'saved {CKPT.name}')
net.eval()

In [ ]:
# Cell 4 · Grad-CAM
# alpha_k = global-average-pooled gradient of the class score wrt feature
# map A^k; the map is the ReLU of the weighted sum of those feature maps.
# Hooks are registered on the last convolutional block.
target_layer = net.features[-1]
_acts, _grads = {}, {}

def fwd_hook(m, i, o): _acts['v'] = o.detach()
def bwd_hook(m, gi, go): _grads['v'] = go[0].detach()

h1 = target_layer.register_forward_hook(fwd_hook)
h2 = target_layer.register_full_backward_hook(bwd_hook)

def gradcam(x, warn=True):
    # x: (1,3,H,W) already normalised and on DEV
    net.zero_grad()
    score = net(x).squeeze()
    score.backward()
    A, G = _acts['v'][0], _grads['v'][0]          # (C,h,w)
    alpha = G.mean(dim=(1, 2))                     # (C,)
    raw = (alpha[:, None, None] * A).sum(0)
    cam = torch.relu(raw)
    # An undertrained or degenerate model can give a weighted sum that is
    # entirely negative, in which case ReLU zeroes the whole map and every
    # downstream alignment score silently becomes 0.0. Detect that rather
    # than reporting zeros as if they were a finding.
    if float(cam.max()) <= 1e-12:
        if warn:
            print('    WARNING: all-negative Grad-CAM weighted sum; '
                  'ReLU produced an empty map.')
            print('    Usually means the model is undertrained or predicts '
                  'one class for everything.')
        # fall back to the absolute map so the figure still shows where
        # the magnitude of influence sits, clearly flagged as such
        cam = raw.abs()
        empty = True
    else:
        empty = False
    cam = cam / (cam.max() + 1e-8)
    cam = torch.nn.functional.interpolate(
        cam[None, None], size=(INPUT_SIZE, INPUT_SIZE),
        mode='bilinear', align_corners=False)[0, 0]
    return cam.cpu().numpy(), float(torch.sigmoid(score)), empty

print('Grad-CAM ready (hooks on', type(target_layer).__name__, ')')

In [ ]:
# Cell 5 · tissue-zone alignment, and what it does and does not mean
#
# The alignment score is the fraction of Grad-CAM activation falling
# inside the k-means cluster mask for a given tissue type:
#
#     A_c = sum_ij( L_ij * M^c_ij ) / sum_ij( L_ij )
#
# READ THIS BEFORE INTERPRETING IT.
# The masks come from the same LAB k-means that notebook 03 showed does
# not identify tissue: it graded 86.3% of healthy skin as severe, and its
# luminance ordering is inverted. A high alignment score therefore says
# the model attends to the same DARK REGIONS the clustering labels
# necrotic (which include shadow, wound depth, hair and darker skin) --
# NOT that it attends to necrotic tissue.
#
# This measures agreement between two procedures that share a failure
# mode. It is reported as an internal-consistency diagnostic only.
from skimage import color
from sklearn.cluster import KMeans

def tissue_masks(path):
    with Image.open(path) as im:
        a = np.asarray(im.convert('RGB').resize((INPUT_SIZE, INPUT_SIZE)),
                       dtype=np.float32) / 255.0
    lab = color.rgb2lab(a).reshape(-1, 3)
    km = KMeans(3, n_init=4, random_state=SEED).fit(lab)
    order = np.argsort(km.cluster_centers_[:, 0])       # darkest first
    remap = np.zeros(3, int)
    for new, old in enumerate(order): remap[old] = new
    lbl = remap[km.labels_].reshape(INPUT_SIZE, INPUT_SIZE)
    return {'necrotic-cluster': lbl == 0,
            'slough-cluster':   lbl == 1,
            'granulation-cluster': lbl == 2}, a

def alignment(cam, mask):
    tot = cam.sum()
    return float((cam * mask).sum() / tot) if tot > 0 else 0.0

print('alignment scoring ready (diagnostic only, see comment above)')

In [ ]:
# Cell 6 · run Grad-CAM over the holdout fold
from collections import defaultdict

# sample per class WITHOUT groupby.apply: that turns the group key into
# the index and drops 'label' from the row, which then breaks r.label
parts = []
for lv in sorted(te.label.unique()):
    g = te[te.label == lv]
    parts.append(g.sample(min(N_EXAMPLES, len(g)), random_state=SEED))
sample = pd.concat(parts).reset_index(drop=True)
print(f'running Grad-CAM on {len(sample)} holdout images...')

records, cams, imgs, n_empty = [], {}, {}, 0
scores = defaultdict(list)
for _, r in sample.iterrows():
    ds = ImgDS([r.path], [r.label])
    x = ds[0][0][None].to(DEV)
    cam, prob, empty = gradcam(x, warn=(len(records) == 0))
    n_empty += int(empty)
    masks, rgb = tissue_masks(r.path)
    a = {k: alignment(cam, m) for k, m in masks.items()}
    records.append(dict(path=r.path, label=int(r.label), prob=prob, **a))
    cams[r.path], imgs[r.path] = cam, rgb
    for k, v in a.items(): scores[f'{k}|label{int(r.label)}'].append(v)

rec = pd.DataFrame(records)
print(f'\nmean activation fraction per cluster (holdout sample):')
for k in ['necrotic-cluster','slough-cluster','granulation-cluster']:
    inf_m = rec[rec.label == 1][k].mean()
    non_m = rec[rec.label == 0][k].mean()
    print(f'  {k:<22} infected {inf_m:.3f}   not infected {non_m:.3f}')
print('\nA cluster covering more of the image will absorb more activation')
print('by area alone, so these are not evidence of clinical targeting.')
if n_empty:
    print(f'\nFLAGGED: {n_empty} of {len(rec)} maps had an all-negative')
    print('weighted sum and fell back to |raw|. Alignment numbers for those')
    print('images reflect magnitude of influence, not positive evidence.')

In [ ]:
# Cell 7 · area-normalised alignment
# The raw fraction above is confounded by cluster size: a cluster covering
# half the image absorbs half the activation from area alone. Dividing by
# the cluster's area fraction gives a lift ratio, where 1.0 means "exactly
# as much activation as its size predicts".
lifts = []
for _, r in rec.iterrows():
    masks, _ = tissue_masks(r.path)
    for k, m in masks.items():
        area = m.mean()
        if area > 0.01:
            lifts.append(dict(label=int(r.label), cluster=k,
                              lift=r[k] / area, area=area))
lift = pd.DataFrame(lifts)

print('activation lift over area (1.0 = proportional to cluster size)')
piv = lift.pivot_table(index='cluster', columns='label', values='lift',
                       aggfunc='mean')
piv.columns = ['not infected', 'infected']
print(piv.to_string(float_format=lambda v: f'{v:.2f}'))
print('\nValues near 1.0 mean the model spreads attention roughly evenly')
print('across clusters, i.e. no preferential targeting of any tissue type.')

In [ ]:
# Cell 8 · figure panel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

show = pd.concat([rec[rec.label == 1].head(3), rec[rec.label == 0].head(3)])
fig, axes = plt.subplots(len(show), 3, figsize=(9, 3 * len(show)))
if len(show) == 1: axes = axes[None, :]

for i, (_, r) in enumerate(show.iterrows()):
    rgb, cam = imgs[r.path], cams[r.path]
    masks, _ = tissue_masks(r.path)
    axes[i,0].imshow(rgb); axes[i,0].set_ylabel(
        f"{'infected' if r.label else 'not infected'}\np={r.prob:.2f}",
        fontsize=9)
    axes[i,1].imshow(rgb); axes[i,1].imshow(cam, cmap='jet', alpha=0.45)
    disp = np.zeros((*cam.shape, 3))
    for j, (k, m) in enumerate(masks.items()):
        disp[m] = [(0.23,0.14,0.09), (0.85,0.64,0.25), (0.78,0.42,0.42)][j]
    axes[i,2].imshow(disp)
    for a in axes[i]: a.set_xticks([]); a.set_yticks([])

axes[0,0].set_title('image', fontsize=10)
axes[0,1].set_title('Grad-CAM', fontsize=10)
axes[0,2].set_title('k-means clusters\n(not validated tissue)', fontsize=9)
fig.suptitle('Grad-CAM attention vs colour clusters\n'
             'Cluster masks are NOT validated tissue labels (notebook 03)',
             fontsize=11)
fig.tight_layout()
fig.savefig(FIGDIR / 'gradcam_panel.png', dpi=200, bbox_inches='tight')
fig.savefig(FIGDIR / 'gradcam_panel.pdf', bbox_inches='tight')
plt.close(fig)
print(f'wrote {(FIGDIR / "gradcam_panel.png").resolve()}')

In [ ]:
# Cell 9 · save and summarise
rec.to_csv(OUT / 'gradcam_alignment.csv', index=False)
summary = dict(
    n_images=int(len(rec)),
    holdout_fold=HOLDOUT,
    mean_alignment={k: float(rec[k].mean()) for k in
                    ['necrotic-cluster','slough-cluster','granulation-cluster']},
    mean_lift=lift.groupby('cluster').lift.mean().round(3).to_dict(),
    caveat=('cluster masks derive from the same LAB k-means shown invalid '
            'in notebook 03 (86.3% of healthy skin graded severe, luminance '
            'ordering inverted). Alignment measures agreement between two '
            'procedures sharing a failure mode, not clinical correctness.'))
with open(OUT / 'gradcam_scores.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'wrote {(OUT / "gradcam_scores.json").resolve()}')

h1.remove(); h2.remove()   # detach hooks

print('\n' + '=' * 58)
print('STAGE 08 COMPLETE')
print('=' * 58)
print(f'  {len(rec)} images, holdout fold {HOLDOUT}')
print(f'  figure: {FIGDIR / "gradcam_panel.png"}')
print('\n  Reported as an attention diagnostic, not as evidence that the')
print('  model attends to clinically correct tissue. The reference masks')
print('  are not validated tissue labels.')
print('\nnext: 09_statistics.ipynb')